# Market Microstructure Analysis

### Objectives

The goal of this project is to analyze the microstructure of a limit order book using high-frequency LOBSTER data.  
Specifically, we aim to:

- Reconstruct and explore the order book at millisecond precision
- Compute key microstructure metrics: mid-price, bid-ask spread, and traded volume
- Calculate the **Order Flow Imbalance (OFI)** to capture supply/demand pressure at the best quotes
- Identify and visualize significant market events and anomalies throughout the trading session

### Structure of the LOBSTER data

- Time: Seconds after midnight with decimal precision of at least milliseconds and up to nanoseconds depending on the period requested

- Event Type:
    - 1: Submission of a new limit order
    - 2: Cancellation (partial deletion of a limit order)
    - 3: Deletion (total deletion of a limit order)
    - 4: Execution of a visible limit order
    - 5: Execution of a hidden limit order
    - 6: Indicates a cross trade, e.g. auction trade
    - 7: Trading halt indicator 

- Order ID: Unique order reference number
- Size: Number of shares
- Price: Dollar price times 10000 (i.e. a stock price of $91.14 is given by 911400)
- Direction:
    - -1: Sell limit order
    - 1: Buy limit order

Note: Execution of a sell (buy) limit order corresponds to a buyer (seller) initiated trade, i.e. buy (sell) trade.

variable explanation.

- Ask Price 1: Level 1 ask price (best ask price)
- Ask Size 1: Level 1 ask volume (best ask volume)
- Bid Price 1: Level 1 bid price (best bid price)
- Bid Size 1: Level 1 bid volume (best bid volume)
- Ask Price 2: Level 2 ask price (second best ask price)
- Ask Size 2: Level 2 ask volume (second best ask volume)


### Packages

In [9]:
import polars as pl
import polars.selectors as cs
import datetime as dt
import hvplot.polars
import holoviews as hv

### Lazy dataframe creation

In [3]:
LEVELS = 1
PRICE_DIVISOR = 10000

col_names_ob = []
for i in range(1, LEVELS + 1):
    col_names_ob.extend([f'ask_p_{i}', f'ask_s_{i}', f'bid_p_{i}', f'bid_s_{i}'])

path_ob = "/Users/thomascgd/Desktop/Lobster Data/LOBSTER_SampleFile_AMZN_2012-06-21_1/AMZN_2012-06-21_34200000_57600000_orderbook_1.csv"
path_msg = "/Users/thomascgd/Desktop/Lobster Data/LOBSTER_SampleFile_AMZN_2012-06-21_1/AMZN_2012-06-21_34200000_57600000_message_1.csv"

df_ob = pl.scan_csv(path_ob,has_header=False, new_columns=col_names_ob)
df_msg = pl.scan_csv(path_msg, has_header=False,
                     new_columns=['time','type','order_id','size','price','direction'])

df = pl.concat([df_msg,df_ob],how="horizontal")

df = df.with_columns(
    pl.duration(seconds="time").alias("duration"),
    (cs.contains("_p_") / PRICE_DIVISOR),
    (pl.col("price") / PRICE_DIVISOR)).with_columns([
    ((pl.col("ask_p_1") + pl.col("bid_p_1")) / 2).alias("mid_price"),
    (pl.col("ask_p_1") - pl.col("bid_p_1")).alias("spread")
])

start_date = dt.datetime(2026, 1, 1)
df = df.with_columns(
    (pl.lit(start_date) + pl.col("duration")).alias("timestamp")
)

In [4]:
df.collect_schema().names()

['time',
 'type',
 'order_id',
 'size',
 'price',
 'direction',
 'ask_p_1',
 'ask_s_1',
 'bid_p_1',
 'bid_s_1',
 'duration',
 'mid_price',
 'spread',
 'timestamp']

Reminder :
- ask_p_1 : ask price level 1
- ask_s_1 : ask size level 1
- bid_p_1 : bid price level 1
- bid_s_1 : bid size level 1

### Analysis of the orders types

In [35]:
df_types = df.select("timestamp","type","size").collect()

In [37]:
df_types.hvplot.hist(y="type")

:Histogram   [type]   (type_count)

*Reminder* : 

- Event Type:
    - 1: Submission of a new limit order
    - 2: Cancellation (partial deletion of a limit order)
    - 3: Deletion (total deletion of a limit order)
    - 4: Execution of a visible limit order
    - 5: Execution of a hidden limit order
    - 6: Indicates a cross trade, e.g. auction trade
    - 7: Trading halt indicator 

In [51]:
order_types_count = {i : df_types.filter(pl.col("type") == i ).select("type").count().item() for i in range(1,8)}
print(f"order_types_count : {order_types_count}")

order_types_size = {i : df_types.filter(pl.col("type") == i ).select("size").sum().item() for i in range(1,8)}
print(f"order_types_size : {order_types_size}")


order_types_count : {1: 27845, 2: 16, 3: 18235, 4: 8974, 5: 2445, 6: 0, 7: 0}
order_types_size : {1: 2670322, 2: 1415, 3: 1666094, 4: 613248, 5: 197507, 6: 0, 7: 0}


In [ ]:
print(f"Number of trades created during the day :  {order_types_count[1]}")
print(f"Number of trades executed during the day :  {order_types_count[4]+order_types_count[5]}")
print(f"Number of total/partial deletion during the day : {order_types_count[2]+order_types_count[3]}")
print(f"Number of trades remaining at least at the end of the day :  {order_types_count[1]-order_types_count[2]-order_types_count[3]} \
      (because we don't know how many trades were remaining from the days before)")
print(f"Number of trades totally/partially deleted during the day + number of trades executed during the day : \
      {order_types_count[2]+order_types_count[3]+order_types_count[4]+order_types_count[5]} ")

Number of trades created during the day :  27845
Number of trades executed during the day :  11419
Number of total/partial deletion during the day : 18251
Number of trades remaining at least at the end of the day :  9594       (because we don't know how many trades were remaining from the days before)
Number of trades totaly/partially deleted during the day + number of trades executed during the day :       29670 


We observe that the number of orders executed or partially/totally cancelled during the day is greater than the number of new limit orders created during the day. This is explained by the possible presence of limit orders from previous days that are still valid (inventory at start of day).

In [61]:
print(f"Fill-or-Cancel Ratio on the number of order : {round((order_types_count[4]+order_types_count[5]) / (order_types_count[2]+order_types_count[3]) * 100, 3)}%")
print(f"Fill-or-Cancel Ratio on the volume : {round((order_types_size[4]+order_types_size[5]) / (order_types_size[2]+order_types_size[3]) * 100, 3)}%")

Fill-or-Cancel Ratio on the number of order : 62.566%
Fill-or-Cancel Ratio on the volume : 48.621%


- Definition : The Fill-to-Cancel Ratio measures the proportion of order book activity that results in an actual trade versus the activity that is withdrawn. It is a vital proxy for understanding market efficiency and the behavior of liquidity providers.

$$\text{Fill-to-Cancel Ratio} = \frac{\sum \text{Volume of Executions (Types 4 \& 5)}}{\sum \text{Volume of Cancellations (Types 2 \& 3)}}$$

- Why it matters for a TraderSignal vs. Noise: 
    - A high ratio indicates "sticky" or "real" liquidity, where participants are committed to their prices. A very low ratio (common in HFT-dominated environments) suggests "phantom liquidity," where orders are canceled as soon as the price moves or a trade is attempted.
    - Execution Strategy: As a trader, a declining ratio during a price move suggests that the liquidity you see in the book might vanish (evanescence) if you try to hit it with a large Market Order.
    - Market Stress Indicator: Sudden drops in this ratio often precede spikes in volatility, as market makers withdraw their quotes to avoid being "picked off" by informed traders.
    - Interpretation of the Data :
        - High Ratio (> 0.1 - 0.5): Robust liquidity. Each cancellation is backed by a significant amount of execution. Typical of "lit" markets during stable periods.
        - Low Ratio (< 0.05): High "Quote Stuffing" or algorithmic probing. For every 100 shares traded, thousands are canceled. This is typical of highly competitive electronic markets like NASDAQ (e.g., AMZN).

### Dataframes creation

In [ ]:
# aggregation of 1 seconde windows
df_1s = (
    df.group_by_dynamic("timestamp", every="1s")
    .agg(
        pl.col("mid_price").first().alias("open"),
        pl.col("mid_price").max().alias("high"),
        pl.col("mid_price").min().alias("low"),
        pl.col("mid_price").last().alias("close"),
        pl.col("size").filter(pl.col("type").is_in([4, 5])).sum().alias("volume"),
        pl.col("spread").mean().alias("avg_spread")
    )
    .collect()
)

# aggregation of 1 milli-seconde windows
df_1ms = (
    df.group_by_dynamic("timestamp", every="1ms")
    .agg(
        pl.col("mid_price").first().alias("open"),
        pl.col("mid_price").max().alias("high"),
        pl.col("mid_price").min().alias("low"),
        pl.col("mid_price").last().alias("close"),
        pl.col("size").filter(pl.col("type").is_in([4, 5])).sum().alias("volume"),
        pl.col("spread").mean().alias("avg_spread")
    )
    .collect() 
)

In [ ]:
print(df_1ms.head())

shape: (5, 7)
┌─────────────────────────┬─────────┬─────────┬─────────┬─────────┬────────┬────────────┐
│ timestamp               ┆ open    ┆ high    ┆ low     ┆ close   ┆ volume ┆ avg_spread │
│ ---                     ┆ ---     ┆ ---     ┆ ---     ┆ ---     ┆ ---    ┆ ---        │
│ datetime[μs]            ┆ f64     ┆ f64     ┆ f64     ┆ f64     ┆ i64    ┆ f64        │
╞═════════════════════════╪═════════╪═════════╪═════════╪═════════╪════════╪════════════╡
│ 2026-01-01 09:30:00.017 ┆ 223.565 ┆ 223.565 ┆ 223.565 ┆ 223.565 ┆ 1      ┆ 0.77       │
│ 2026-01-01 09:30:00.189 ┆ 223.88  ┆ 223.88  ┆ 223.88  ┆ 223.88  ┆ 0      ┆ 0.14       │
│ 2026-01-01 09:30:00.190 ┆ 223.85  ┆ 223.85  ┆ 223.85  ┆ 223.85  ┆ 47     ┆ 0.2        │
│ 2026-01-01 09:30:00.372 ┆ 223.85  ┆ 223.85  ┆ 223.85  ┆ 223.85  ┆ 100    ┆ 0.2        │
│ 2026-01-01 09:30:00.375 ┆ 223.85  ┆ 223.85  ┆ 223.85  ┆ 223.85  ┆ 100    ┆ 0.2        │
└─────────────────────────┴─────────┴─────────┴─────────┴─────────┴────────┴──────────

In [ ]:
df_1ms.select(pl.all().exclude("timestamp")).describe()

statistic,open,high,low,close,volume,avg_spread
str,f64,f64,f64,f64,f64,f64
"""count""",37081.0,37081.0,37081.0,37081.0,37081.0,37081.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",222.709958,222.711406,222.708544,222.710025,21.864432,0.135803
"""std""",1.362231,1.36257,1.36186,1.362223,83.68443,0.06599
"""min""",220.52,220.52,220.52,220.52,0.0,0.01
"""25%""",221.285,221.29,221.285,221.285,0.0,0.0925
"""50%""",222.665,222.67,222.665,222.665,0.0,0.13
"""75%""",223.85,223.855,223.85,223.85,0.0,0.17
"""max""",226.025,226.025,226.015,226.015,4018.0,0.77


## Plots

### Price plot

In [ ]:
p1 = df_1s.hvplot.line(x="timestamp",y="open",width=600)
p2 = hv.VLines([dt.datetime(year=2026,month=1,day=1,hour=10,minute=52),
                dt.datetime(year=2026,month=1,day=1,hour=12,minute=25)]).opts(
                    color='red', line_width = 1.5, line_dash = "dotted"
                )
p1 * p2

:Overlay
   .Curve.I  :Curve   [timestamp]   (open)
   .VLines.I :VLines   [x]

On this graph, we can see several abrut changes in price, notably at 10h52 and 12h25.

###  Spread bid-ask plot 

In [ ]:
df_1s.hvplot.line(x="timestamp",y="avg_spread",width=600)

:Curve   [timestamp]   (avg_spread)

You can see in the second graph that the wider spreads are at the beginning of the day, just after the market has opened. 
This seems to be caused by the **adverse selection**.

*Reminder* : 
**Adverse selection** is a phenomem caused by the fact that some investors have information that the market marker doesn't have yet. As a consequence, the market market doesn't trade the asset at the real price and it could cost him money. In order to reduce that risk, at the beginning of the day, the spread is wider and the volume of transactions is limited in order to diminish the risk of adverse selection. Howerver this diminish the liquidity of the asset.

### Violin plot

In [ ]:
df_with_hour = df_1s.with_columns(pl.col("timestamp").dt.hour().alias("hour"))
df_with_hour.hvplot.violin(y="avg_spread", by="hour", width=600)

:Violin   [hour]   (avg_spread)

On this graph, we can see that the different price distributions throughout the day. We can contast that the distribution with the most variance is the first distribution (i.e the first hour of the day), this reflects the adverse selection phenomem.

### Analysis of volume

In [ ]:
v1 = df_1ms.hvplot.line(x="timestamp",y="volume")
v2 = df_1ms.hvplot.line(x="timestamp",y="open")
v3 = hv.VLines([dt.datetime(year=2026,month=1,day=1,hour=10,minute=3),
                dt.datetime(year=2026,month=1,day=1,hour=10,minute=52),
                dt.datetime(year=2026,month=1,day=1,hour=11,minute=7),
                dt.datetime(year=2026,month=1,day=1,hour=14,minute=7),
                dt.datetime(year=2026,month=1,day=1,hour=15,minute=20),
                dt.datetime(year=2026,month=1,day=1,hour=15,minute=50),
                dt.datetime(year=2026,month=1,day=1,hour=15,minute=52),
                ]).opts(
                    color='red', line_width = 1.5, line_dash = "dotted"
                )
figure_volume = (v1 + v2 * v3).cols(1)
figure_volume


:Layout
   .Curve.I   :Curve   [timestamp]   (volume)
   .Overlay.I :Overlay
      .Curve.I  :Curve   [timestamp]   (open)
      .VLines.I :VLines   [x]

- Lines representing the 7 biggest volumes pics : 
    - Line 1 : Time : 10h03 | Volume : 2100
    - Line 2 : Time : 10h52 | Volume : 1888
    - Line 3 : Time : 11h07 | Volume : 2524
    - Line 4 : Time : 14h07 | Volume : 4018
    - Line 5 : Time : 15h20 | Volume : 2268
    - Line 6 : Time : 15h50 | Volume : 2440
    - Line 7 : Time : 15h52 | Volume : 2340

We can see that the volume peak is at 14h07, but the price at this time doesn't move a lot. 
And when the price move sharply, the volume is not always above the average. (example : 12h23 - 12h27)
That is very interesting. We need to investigate this.

In [ ]:
df_volumes = df.select(["timestamp","type","size"]).collect()

In [ ]:
volume_VL1 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,10,3,0),dt.datetime(2026,1,1,10,3,59))).hvplot.hist(y="size", by="type",title = "Line 1")
volume_VL2 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,10,52,0),dt.datetime(2026,1,1,10,52,59))).hvplot.hist(y="size", by="type",title = "Line 2")
volume_VL3 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,11,7,0),dt.datetime(2026,1,1,11,7,59))).hvplot.hist(y="size", by="type",title = "Line 3")
volume_VL4 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,14,7,0),dt.datetime(2026,1,1,14,7,59))).hvplot.hist(y="size", by="type",title = "Line 4")
volume_VL5 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,15,20,0),dt.datetime(2026,1,1,15,20,59))).hvplot.hist(y="size", by="type",title = "Line 5")
volume_VL6 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,15,50,0),dt.datetime(2026,1,1,15,50,59))).hvplot.hist(y="size", by="type",title = "Line 6")
volume_VL7 = df_volumes.filter(pl.col("type").is_in([4,5]) & pl.col("timestamp").is_between(dt.datetime(2026,1,1,15,52,0),dt.datetime(2026,1,1,15,52,59))).hvplot.hist(y="size", by="type",title = "Line 7")
(volume_VL1 + volume_VL2 + volume_VL3 + volume_VL4 + volume_VL5 + volume_VL6 + volume_VL7).opts(shared_axes=False, width=500).cols(1)

:Layout
   .NdOverlay.I   :NdOverlay   [type]
      :Histogram   [size]   (size_count)
   .NdOverlay.II  :NdOverlay   [type]
      :Histogram   [size]   (size_count)
   .NdOverlay.III :NdOverlay   [type]
      :Histogram   [size]   (size_count)
   .NdOverlay.IV  :NdOverlay   [type]
      :Histogram   [size]   (size_count)
   .NdOverlay.V   :NdOverlay   [type]
      :Histogram   [size]   (size_count)
   .NdOverlay.VI  :NdOverlay   [type]
      :Histogram   [size]   (size_count)
   .NdOverlay.VII :NdOverlay   [type]
      :Histogram   [size]   (size_count)

*Reminder* : 
- Event type :
    - 4: Execution of a visible limit order
    - 5: Execution of a hidden limit order

### Order Flow Imbalance (OFI)

In [ ]:
df_ofi = df.filter(pl.col("type").is_in([4, 5])).with_columns([
    pl.when(pl.col("bid_p_1") >= pl.col("bid_p_1").shift(1))
      .then(pl.col("bid_s_1"))
      .otherwise(-pl.col("bid_s_1"))
      .alias("e_bid"),
    
    pl.when(pl.col("ask_p_1") <= pl.col("ask_p_1").shift(1))
      .then(pl.col("ask_s_1"))
      .otherwise(-pl.col("ask_s_1"))
      .alias("e_ask")
]).with_columns(
    (pl.col("e_bid") - pl.col("e_ask")).alias("ofi")
).collect()

df_ofi_1s = df_ofi.group_by_dynamic("timestamp", every="1s").agg(
    pl.col("ofi").mean(),
    pl.col("mid_price").last()
)

In [ ]:
print(df_ofi_1s.head())

shape: (5, 3)
┌─────────────────────┬─────────────┬───────────┐
│ timestamp           ┆ ofi         ┆ mid_price │
│ ---                 ┆ ---         ┆ ---       │
│ datetime[μs]        ┆ f64         ┆ f64       │
╞═════════════════════╪═════════════╪═══════════╡
│ 2026-01-01 09:30:00 ┆ -368.068966 ┆ 223.895   │
│ 2026-01-01 09:30:01 ┆ 110.0       ┆ 224.04    │
│ 2026-01-01 09:30:02 ┆ 80.0        ┆ 224.04    │
│ 2026-01-01 09:30:04 ┆ 0.0         ┆ 224.01    │
│ 2026-01-01 09:30:05 ┆ 100.0       ┆ 223.97    │
└─────────────────────┴─────────────┴───────────┘


In [ ]:
df_ofi_1s.hvplot.line(x="timestamp",y=["mid_price","ofi"],subplots=True,shared_axes=False).cols(1)

:NdLayout   [Variable]
   :Curve   [timestamp]   (value)

In [ ]:
df_ofi.filter((pl.col("ofi") > 10000) & (pl.col("timestamp").dt.time() < dt.time(11,30,47)))

time,type,order_id,size,price,direction,ask_p_1,ask_s_1,bid_p_1,bid_s_1,duration,mid_price,spread,timestamp,e_bid,e_ask,ofi
f64,i64,i64,i64,f64,i64,f64,i64,f64,i64,duration[μs],f64,f64,datetime[μs],i64,i64,i64
41326.144326,4,116394874,100,223.93,-1,223.96,100,223.9,33970,11h 28m 46s 144325µs,223.93,0.06,2026-01-01 11:28:46.144325,33970,-100,34070
41326.15685,4,115416710,100,223.96,-1,223.97,300,223.9,33970,11h 28m 46s 156850µs,223.935,0.07,2026-01-01 11:28:46.156850,33970,-300,34270
41326.15685,5,0,23,223.96,-1,223.97,300,223.9,33970,11h 28m 46s 156850µs,223.935,0.07,2026-01-01 11:28:46.156850,33970,300,33670
41326.157098,5,0,77,223.96,-1,223.97,200,223.9,33970,11h 28m 46s 157097µs,223.935,0.07,2026-01-01 11:28:46.157097,33970,200,33770
41326.164945,4,115235438,200,223.97,-1,224.02,123,223.9,33970,11h 28m 46s 164944µs,223.96,0.12,2026-01-01 11:28:46.164944,33970,-123,34093
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
41406.023988,4,117307354,100,223.98,-1,224.0,17,223.95,29700,11h 30m 6s 23988µs,223.975,0.05,2026-01-01 11:30:06.023988,29700,17,29683
41406.024158,4,117148778,17,224.0,-1,224.01,200,223.95,29700,11h 30m 6s 24158µs,223.98,0.06,2026-01-01 11:30:06.024158,29700,-200,29900
41406.024158,4,117180769,83,224.01,-1,224.01,117,223.95,29700,11h 30m 6s 24158µs,223.98,0.06,2026-01-01 11:30:06.024158,29700,117,29583
